In [1]:
import tensorflow as tf
import time
import numpy as np
import os
import copy
import pickle
import argparse
import utilityarm as utility
import pandas as pd
from sklearn.metrics import *
import tensorflow.keras.backend as K

In [2]:
import tensorflow.compat.v1 as tf

tf.disable_v2_behavior() 


class FairAdvBPR:

    def __init__(self, sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count):
       
        self.dataname = dict_args['dataname']

        self.key_type = key_type
        self.user_type_list = user_type_list
        self.item_type_count = item_type_count
        self.layers = dict_args['layers']
        self.sess = sess
        
        self.num_cols = len(train_df['item_id'].unique())
        self.num_rows = len(train_df['user_id'].unique())

        self.hidden_neuron = dict_args['hidden_neuron']
        self.neg = dict_args['neg']
        self.batch_size = dict_args['batch_size']

        self.train_df = train_df
        self.vali_df = test_df
        self.num_train = len(self.train_df)
        self.num_vali = len(self.vali_df)

        self.train_epoch = dict_args['train_epoch']
        self.train_epoch_a = dict_args['train_epoch_a']

        self.lr_r = dict_args['lr_r'] # learning rate
        self.lr_a = dict_args['lr_a'] # learning rate
        self.alpha = dict_args['alpha'] # learning rate
        self.optimizer_method = dict_args['optimizer_method']
        self.display_step = dict_args['display_step']
        
        self.type_error_weight = type_error_weight
        self.num_type = dict_args['num_type']
        
        self.user_type = user_type
        self.type_count_list = []
        for k in range(self.num_type):
            self.type_count_list.append(np.sum(user_type[:,k]))

        
        self.reg = dict_args['reg'] # regularization term trade-off
        self.reg_s = dict_args['reg_s']

        print('**********fairAdvBPR**********')
        #print(self.args)
        self._prepare_model()

    def loadmodel(self, saver, checkpoint_dir):
        ckpt = tf.train.get_checkpoint_state(checkpoint_dir)
        if ckpt and ckpt.model_checkpoint_path:
            ckpt_name = os.path.basename(ckpt.model_checkpoint_path)
            saver.restore(self.sess, os.path.join(checkpoint_dir, ckpt_name))
            return True
        else:
            return False
        
    def run(self):
        init = tf.global_variables_initializer()
        self.sess.run(init)

        saver = tf.train.Saver([self.P, self.Q])
        self.loadmodel(saver, "./"+self.dataname+"/BPR_check_points")

        for epoch_itr in range(1, self.train_epoch + 1 + self.train_epoch_a):
            self.train_model(epoch_itr)
            if epoch_itr % self.display_step == 0:
                self.test_model(epoch_itr)
        return self.make_records()

    def _prepare_model(self):
        with tf.name_scope("input_data"):
            self.user_input = tf.placeholder(tf.int32, shape=[None, 1], name="user_input")
            self.item_input_pos = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_pos")
            self.item_input_neg = tf.placeholder(tf.int32, shape=[None, 1], name="item_input_neg")

            self.input_user_type = tf.placeholder(dtype=tf.float32, shape=[None, self.num_type]
                                                   , name="input_user_type")
            self.input_user_error_weight = tf.placeholder(dtype=tf.float32, shape=[None, 1]
                                                          , name="input_user_error_weight")

        with tf.variable_scope("BPR", reuse=tf.AUTO_REUSE):
            self.P = tf.get_variable(name="P",
                                     initializer=tf.truncated_normal(shape=[self.num_rows, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
            self.Q = tf.get_variable(name="Q",
                                     initializer=tf.truncated_normal(shape=[self.num_cols+1, self.hidden_neuron], mean=0,
                                                                     stddev=0.03), dtype=tf.float32)
        para_r = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="BPR")

        with tf.variable_scope("Adversarial", reuse=tf.AUTO_REUSE):
            num_layer = len(self.layers)
            adv_W = []
            adv_b = []
            for l in range(num_layer):
                if l == 0:
                    in_shape = 21
                else:
                    in_shape = self.layers[l - 1]
                adv_W.append(tf.get_variable(name="adv_W" + str(l),
                                             initializer=tf.truncated_normal(shape=[in_shape, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
                adv_b.append(tf.get_variable(name="adv_b" + str(l),
                                             initializer=tf.truncated_normal(shape=[1, self.layers[l]],
                                                                             mean=0, stddev=0.03), dtype=tf.float32))
            adv_W_out = tf.get_variable(name="adv_W_out",
                                        initializer=tf.truncated_normal(shape=[self.layers[-1], self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)

            adv_b_out = tf.get_variable(name="adv_b_out",
                                        initializer=tf.truncated_normal(shape=[1, self.num_type],
                                                                        mean=0, stddev=0.03), dtype=tf.float32)
        para_a = tf.get_collection(tf.GraphKeys.GLOBAL_VARIABLES, scope="Adversarial")

        p = tf.reduce_sum(tf.nn.embedding_lookup(self.P, self.user_input), 1)
        q_neg = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_neg), 1)
        q_pos = tf.reduce_sum(tf.nn.embedding_lookup(self.Q, self.item_input_pos), 1)

        predict_pos = tf.reduce_sum(p * q_pos, 1)
        predict_neg = tf.reduce_sum(p * q_neg, 1)

        r_cost1 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg)))
        r_cost2 = self.reg * 0.5 * (self.l2_norm(self.P) + self.l2_norm(self.Q))  # regularization term
#         pred = tf.matmul(self.P, tf.transpose(self.Q))
#         self.s_mean = tf.reduce_mean(pred, axis=1)
#         self.s_std = tf.keras.backend.std(pred, axis=1)
#         self.s_cost = tf.reduce_sum(tf.square(self.s_mean) + tf.square(self.s_std) - 2 * tf.log(self.s_std) - 1)#additional regularization
        self.s_mean = 0
        self.s_std = 0
        self.s_cost = 0
        
#        print('input_user_type ',self.input_user_type[:,1].shape)
#         usertype = self.input_user_type[]
#         print('shape',usertype.shape)
#         const_user_type1 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg) * self.input_user_type[:,0]))
#         const_user_type2 = tf.reduce_sum(tf.nn.softplus(-(predict_pos - predict_neg) * self.input_user_type[:,1]))
        
#        const = K.sqrt(K.square(const_user_type1 - const_user_type2))
        
#        self.r_cost = r_cost1 + r_cost2 + 1 * const + self.reg_s * 0.5 * self.s_cost
        self.r_cost = r_cost1 + r_cost2 
        
        print('shape q pos',q_pos.shape)
        print('shape p',p.shape)
        print('shape predict pos ', predict_pos.shape)
        
        adv_last = tf.reshape(predict_pos, [tf.shape(self.input_user_type)[0], 1])
        print('shape adv_last ', adv_last.shape)
        adv_last = tf.concat([adv_last, q_pos], 1)
        
        for l in range(num_layer):
            adv = tf.nn.relu(tf.matmul(adv_last, adv_W[l]) + adv_b[l])
            adv_last = adv
        self.adv_output = tf.nn.sigmoid(tf.matmul(adv_last, adv_W_out) + adv_b_out)
 #       self.a_cost = tf.reduce_sum(tf.square(self.adv_output - self.input_user_type) * self.input_user_error_weight)
        self.a_cost = -tf.reduce_sum(self.input_user_type * tf.math.log(self.adv_output))

    
                  

 #       self.all_cost = self.r_cost - self.alpha * self.a_cost  # the loss function
        self.all_cost = self.r_cost - 10000 * self.a_cost  # the loss function

        with tf.variable_scope("Optimizer", reuse=tf.AUTO_REUSE):
            self.r_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.r_cost, var_list=para_r)
            self.a_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_a).minimize(self.a_cost, var_list=para_a)
            self.all_optimizer = tf.train.AdamOptimizer(learning_rate=self.lr_r).minimize(self.all_cost, var_list=para_r)


    def train_model(self, itr):
        NS_start_time = time.time() * 1000.0
        epoch_r_cost = 0.0
        epoch_s_cost = 0.0
        epoch_s_mean = 0.0
        epoch_s_std = 0.0
        epoch_a_cost = 0.0
        accuracy_adv_pre_temp = 0.0
        accuracy_adv_post_temp = 0.0
        accuracy_adv_pre =[]
        accuracy_adv_post =[]
        epoch_adv_acc_pre = 0.0
        epoch_adv_acc_post = 0.0
        num_sample, user_list, item_pos_list, item_neg_list = utility.negative_sample(self.train_df, self.num_rows,
                                                                                      self.num_cols, self.neg)
        NS_end_time = time.time() * 1000.0

        start_time = time.time() * 1000.0
        num_batch = int(num_sample / float(self.batch_size)) + 1
        random_idx = np.random.permutation(num_sample)
        for i in range(num_batch):
            # get the indices of the current batch
            if i == num_batch - 1:
                batch_idx = random_idx[i * self.batch_size:]
            elif i < num_batch - 1:
                batch_idx = random_idx[(i * self.batch_size):((i + 1) * self.batch_size)]
            
            accuracy_adv_pre_temp = 0.0
            accuracy_adv_post_temp = 0.0
            if itr > self.train_epoch:
                #random_idx_a = np.random.permutation(num_sample)
                #print("boucle adversarial debut-- num batch ",i)
                #for j in range(num_batch):
#                 if j == num_batch - 1:
#                 batch_idx_a = random_idx_a[j * self.batch_size:]
#                 elif j < num_batch - 1:
#                 batch_idx_a = random_idx_a[(j * self.batch_size):((j + 1) * self.batch_size)]
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_a_cost, adv_output = self.sess.run(  # do the optimization by the minibatch
                        [self.a_optimizer, self.a_cost, self.adv_output],
                feed_dict={self.user_input: user_list[batch_idx, :],
                                   self.item_input_pos: item_pos_list[batch_idx, :],
                                   self.item_input_neg: item_neg_list[batch_idx, :],
                                   self.input_user_type: self.user_type[user_idx_list,:],
                                   self.input_user_error_weight: self.type_error_weight[user_idx_list,:]})
                epoch_a_cost += tmp_a_cost
                    
                g = tf.Graph()
                with g.as_default():
                    logits = adv_output
                      
                    labels = self.user_type[user_idx_list,:]
                      
                    acc, acc_op = tf.compat.v1.metrics.accuracy(labels, logits)
                    global_init = tf.compat.v1.global_variables_initializer()
                    local_init = tf.compat.v1.local_variables_initializer()
                sess = tf.compat.v1.Session(graph=g)
                sess.run([global_init, local_init])
                acc, acc_op = sess.run([acc, acc_op])
                    #print('post adversarial -- adversaire ready ',acc_op)
                accuracy_adv_pre_temp += acc_op
                sess.close()
                accuracy_adv_pre.append(accuracy_adv_pre_temp)
                print('post adversarial -- adversaire ready after one adv epoch training ',accuracy_adv_pre_temp)
                
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost = self.sess.run(  # do the optimization by the minibatch
                    [self.all_optimizer, self.all_cost],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
                #mesure perf adv post traininf of main model
#                 for j in range(num_batch):
#                     if j == num_batch - 1:
#                         batch_idx_a = random_idx_a[j * self.batch_size:]
#                     elif j < num_batch - 1:
#                         batch_idx_a = random_idx_a[(j * self.batch_size):((j + 1) * self.batch_size)]
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _,_ = self.sess.run(  # do the optimization by the minibatch
                         [self.a_optimizer, self.a_cost], feed_dict={self.user_input: user_list[batch_idx, :],
                                    self.item_input_pos: item_pos_list[batch_idx, :],
                                    self.item_input_neg: item_neg_list[batch_idx, :],
                                    self.input_user_type: self.user_type[user_idx_list,:],
                                    self.input_user_error_weight: self.type_error_weight[user_idx_list,:]})
        
                
                adv_output_post = self.sess.run(  # do the optimization by the minibatch
                        [ self.adv_output],
                feed_dict={self.user_input: user_list[batch_idx, :],
                                   self.item_input_pos: item_pos_list[batch_idx, :],
                                   self.item_input_neg: item_neg_list[batch_idx, :],
                                   self.input_user_type: self.user_type[user_idx_list,:],
                                   self.input_user_error_weight: self.type_error_weight[user_idx_list,:]})
                    
                g = tf.Graph()
                with g.as_default():
                    labels2 = self.user_type[user_idx_list,:] 
                    logits2 = adv_output_post                    
                    acc, acc_op = tf.compat.v1.metrics.accuracy(labels2, logits2[0])
                    global_init = tf.compat.v1.global_variables_initializer()
                    local_init = tf.compat.v1.local_variables_initializer()
                sess = tf.compat.v1.Session(graph=g)
                sess.run([global_init, local_init])
                acc, acc_op = sess.run([acc, acc_op])
                    #print('post main model training -- adversaire dejoue', acc_op)
                accuracy_adv_post_temp += acc_op
                sess.close()
                accuracy_adv_post.append(accuracy_adv_post_temp)
                print('post adversarial -- adversaire dejoue after one re-training of main model ',accuracy_adv_post_temp)
                
                
                print("boucle adversarial fin")
            else:
                user_idx_list = ((user_list[batch_idx, :]).reshape((len(batch_idx)))).tolist()
                _, tmp_r_cost = self.sess.run(  # do the optimization by the minibatch
                    [self.r_optimizer, self.r_cost],
                    feed_dict={self.user_input: user_list[batch_idx, :],
                               self.item_input_pos: item_pos_list[batch_idx, :],
                               self.item_input_neg: item_neg_list[batch_idx, :],
                               self.input_user_type: self.user_type[user_idx_list, :],
                               self.input_user_error_weight: self.type_error_weight[user_idx_list, :]})
                epoch_r_cost += tmp_r_cost
    
        epoch_a_cost /= num_batch
        epoch_adv_acc_pre = sum(accuracy_adv_pre)/num_batch
        epoch_adv_acc_post = sum(accuracy_adv_post)/num_batch
        
        filename = './const_IembfairAdvBPR_new_results/epoch'+ str(itr) +'_pre_' + self.dataname + '_accAdvpre.npy'
        os.makedirs(os.path.dirname(filename), exist_ok=True)           
        with open(filename, "wb") as f:
                np.save(f, epoch_adv_acc_pre) 
        filename1 = './const_IembfairAdvBPR_new_results/epoch'+ str(itr) +'_post_' + self.dataname + '_accAdvpost.npy'
        os.makedirs(os.path.dirname(filename1), exist_ok=True)           
        with open(filename1, "wb") as f:
                np.save(f, epoch_adv_acc_post) 
                
        if itr % self.display_step == 0:
            print ("Training //", "Epoch %d //" % itr, " Total r_cost = %.5f" % epoch_r_cost,
                   " Total a_cost = %.5f" % epoch_a_cost,
                   "total pre adversarial accuracy = %.5f" % epoch_adv_acc_pre,
                   "total post adversarial accuracy = %.5f" % epoch_adv_acc_post,
                   "Training time : %d ms" % (time.time() * 1000.0 - start_time),
                   "negative Sampling time : %d ms" % (NS_end_time - NS_start_time),
                   "negative samples : %d" % (num_sample))
       
    def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
        if itr % self.display_step == 0:
            start_time = time.time() * 1000.0
            P, Q = self.sess.run([self.P, self.Q])
            Rec = np.matmul(P, Q.T)

            [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
#             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_genre, self.item_genre_list,
#                                      self.user_genre_count)
            utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
            auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
            print("AUC global is: ", auc)
            
            filename = './const_IembfairAdvBPR_new_results/epoch'+ str(itr) +'_Rec_' + self.dataname + '_constfairAdvBPR.npy'
            os.makedirs(os.path.dirname(filename), exist_ok=True)           
            with open(filename, "wb") as f:
                np.save(f, Rec)
                

            

    def make_records(self):  # record all the results' details into files
        P, Q = self.sess.run([self.P, self.Q])
        Rec = np.matmul(P, Q.T)

        [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
        return precision, recall, f_score, NDCG, Rec

#     def test_model(self, itr):  # calculate the cost and rmse of testing set in each epoch
#         if itr % self.display_step == 0:
#             start_time = time.time() * 1000.0
#             P, Q = self.sess.run([self.P, self.Q])
#             Rec = np.matmul(P, Q.T)

#             [precision, recall, f_score, NDCG] = utility.test_model_all(Rec, self.vali_df, self.train_df)
# #             utility.ranking_analysis(Rec, self.vali_df, self.train_df, self.key_type, self.user_type_list,
# #                                      self.item_type_count)
#             utility.test_model_per_user_type(Rec, self.vali_df, self.train_df, self.user_type_list, self.key_type)
#             auc = utility.auc_per_user(Rec, self.vali_df, self.train_df)
#             print("AUC global is: ", auc)
#             print (
#                 "Testing //", "Epoch %d //" % itr,
#                 "Testing time : %d ms" % (time.time() * 1000.0 - start_time))
#             print("=" * 200)


    @staticmethod
    def l2_norm(tensor):
        return tf.reduce_sum(tf.square(tensor))


Instructions for updating:
non-resource variables are not supported in the long term


In [3]:

#optimizer_method = ['Adam', 'Adadelta', 'Adagrad', 'RMSProp', 'GradientDescent','Momentum'], default='Adam')


train_epoch = 6
train_epoch_a = 10 #default 20
display_step = 1
lr_r = 0.01
lr_a = 0.005
reg = 0.1
reg_s = 30
alpha = 1000
optimizer_method = 'Adam'
hidden_neuron = 20
n = 1
neg = 5
batch_size = 1024
#layers = [50, 50, 50, 50]
layers = [64]
dataname = 'ml1m-6'

In [4]:
dict_args =  {"train_epoch": train_epoch,
              "train_epoch_a": train_epoch_a,
            "display_step":display_step,
            "lr_r":lr_r,
            "lr_a":lr_a,
            "reg":reg,
            "reg_s":reg_s,
            "alpha":alpha,
            "optimizer_method":optimizer_method,
            "hidden_neuron":hidden_neuron,
            "n":n,
            "neg":neg,
            "batch_size":batch_size,
            "layers":layers,
            "dataname":dataname}
dict_args

{'train_epoch': 6,
 'train_epoch_a': 10,
 'display_step': 1,
 'lr_r': 0.01,
 'lr_a': 0.005,
 'reg': 0.1,
 'reg_s': 30,
 'alpha': 1000,
 'optimizer_method': 'Adam',
 'hidden_neuron': 20,
 'n': 1,
 'neg': 5,
 'batch_size': 1024,
 'layers': [64],
 'dataname': 'ml1m-6'}

In [5]:
with open('./training_df.pkl', 'rb') as f:
    train_df = pickle.load(f,encoding='latin1')

# with open('./' + dataname + '/valiing_df.pkl', 'rb') as f:
#     vali_df = pickle.load(f,encoding='latin1')  # for validation
    
with open('./testing_df.pkl', 'rb') as f:
    test_df = pickle.load(f,encoding='latin1')  # for validation
# vali_df = pickle.load(open('./' + dataname + '/testing_df.pkl'))  # for testing

with open('./key_type.pkl', 'rb') as f:
    key_type = pickle.load(f,encoding='latin1')
    
with open('./user_idd_type_list.pkl', 'rb') as f:
    user_idd_type_list = pickle.load(f,encoding='latin1')
    
with open('./type_user_vector.pkl', 'rb') as f:
    type_user_vector = pickle.load(f,encoding='latin1')

with open('./type_count.pkl', 'rb') as f:
    type_count = pickle.load(f,encoding='latin1')
    
with open('./item_type_count.pkl', 'rb') as f:
    item_type_count = pickle.load(f,encoding='latin1')

In [6]:
train_df.head(20)

,user_id,item_id,rating
0,1,1,3
1,2,2,1
2,3,3,2
3,4,4,1
4,6,6,2
5,7,7,5
6,8,8,3
7,9,9,3
8,10,10,2
9,11,11,5


In [7]:
train_df.shape

(80146, 3)

In [8]:
test_df.head(20)

,user_id,item_id,rating
0,0,0,3
1,5,5,4
2,12,12,5
3,13,13,3
4,17,17,2
5,20,20,4
6,24,25,2
7,30,31,4
8,35,35,1
9,40,40,4


In [9]:
test_df.shape

(19577, 3)

In [10]:
print(len(user_idd_type_list))

943


In [11]:
user_idd_type_list

[['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['F'],
 ['F'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['F'],
 ['M'],
 ['M'],
 ['M'],
 ['M'],
 ['F'],
 ['F'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],
 ['M'],
 ['F'],
 ['M'],


In [12]:
print(type_user_vector['F'].shape)

(1, 943)


In [13]:
type_user_vector

{'M': array([[1., 0., 1., 1., 1., 1., 1., 0., 1., 1., 0., 1., 1., 1., 0., 1.,
         0., 1., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0.,
         1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1., 1.,
         1., 1., 0., 1., 1., 0., 1., 1., 0., 0., 1., 1., 1., 1., 0., 1.,
         1., 1., 0., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 1., 0.,
         1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 1., 0., 1., 1.,
         0., 0., 0., 1., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0.,
         0., 1., 0., 1., 1., 0., 1., 1., 0., 1., 1., 0., 1., 1., 1., 1.,
         1., 0., 1., 1., 1., 1., 0., 0., 1., 1., 1., 1., 0., 1., 1., 1.,
         0., 1., 1., 1., 1., 1., 0., 1., 1., 0., 0., 1., 1., 1., 0., 1.,
         0., 1., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 1., 0., 1.,
         1., 1., 0., 0., 1., 1., 1., 1., 1., 1., 0., 1., 1., 1., 1., 0.,
         1., 1., 1., 1., 1., 0., 1., 0., 1., 0., 0., 0., 1., 0., 1., 0.,
         1., 0., 1., 1., 1., 1., 1., 1., 0., 1

In [14]:
len(item_type_count)

1473

In [15]:
item_type_count

[{'M': 605, 'F': 247},
 {'M': 500, 'F': 206},
 {'M': 666, 'F': 270},
 {'M': 630, 'F': 249},
 {'M': 592, 'F': 257},
 {'M': 531, 'F': 251},
 {'M': 523, 'F': 240},
 {'M': 623, 'F': 251},
 {'M': 572, 'F': 231},
 {'M': 585, 'F': 232},
 {'M': 492, 'F': 212},
 {'M': 618, 'F': 248},
 {'M': 429, 'F': 217},
 {'M': 640, 'F': 261},
 {'M': 604, 'F': 256},
 {'M': 655, 'F': 254},
 {'M': 643, 'F': 247},
 {'M': 567, 'F': 220},
 {'M': 656, 'F': 267},
 {'M': 663, 'F': 271},
 {'M': 635, 'F': 254},
 {'M': 638, 'F': 252},
 {'M': 509, 'F': 236},
 {'M': 493, 'F': 204},
 {'M': 404, 'F': 177},
 {'M': 520, 'F': 231},
 {'M': 537, 'F': 226},
 {'M': 636, 'F': 260},
 {'M': 630, 'F': 250},
 {'M': 494, 'F': 221},
 {'M': 595, 'F': 249},
 {'M': 442, 'F': 201},
 {'M': 570, 'F': 244},
 {'M': 554, 'F': 212},
 {'M': 524, 'F': 227},
 {'M': 665, 'F': 272},
 {'M': 551, 'F': 232},
 {'M': 648, 'F': 262},
 {'M': 617, 'F': 260},
 {'M': 645, 'F': 266},
 {'M': 590, 'F': 231},
 {'M': 647, 'F': 270},
 {'M': 653, 'F': 270},
 {'M': 567,

In [16]:
print(type_count)

{'M': 670, 'F': 273}


In [17]:
num_item = len(train_df['item_id'].unique())
num_user = len(train_df['user_id'].unique())
num_type = len(key_type)
print('items number : ',num_item)
print('users number : ',num_user)
print('user types : ',key_type)


items number :  1472
users number :  943
user types :  ['M', 'F']


In [18]:
dict_args["num_type"] = len(key_type)

In [19]:
user_type_list = [] #preprocessing to be sure that user types are really the right ones armielle 
for u in range(num_user):
    gl = user_idd_type_list[u]
    tmp = []
    for g in gl:
        if g in key_type:
            tmp.append(g)
    user_type_list.append(tmp)

print(len(user_type_list))

943


In [20]:
# genreate user_type matrix
user_type = np.zeros((num_user, num_type))
for u in range(num_user):
    gl = user_type_list[u]
    for k in range(num_type):
        if key_type[k] in gl:
            user_type[u, k] = 1.0

In [21]:
len(user_type_list)

943

In [22]:
print('*' * 50)
print('number of positive feedback: ' + str(len(train_df)))
print('estimated number of training samples: ' + str(neg * len(train_df)))
print('*' * 50)

**************************************************
number of positive feedback: 80146
estimated number of training samples: 400730
**************************************************


In [23]:
type_count_mean_reciprocal = []
for k in key_type:
    type_count_mean_reciprocal.append(1.0 / type_count[k])
type_count_mean_reciprocal = (np.array(type_count_mean_reciprocal)).reshape((num_type, 1))
type_error_weight = np.dot(user_type, type_count_mean_reciprocal)


In [24]:
# generate user_type matrix
type_user_indicator = np.zeros((num_type, num_user))

for k in range(num_type):
    type_user_indicator[k,:] = type_user_vector[key_type[k]]


In [25]:
precision = np.zeros(4)
recall = np.zeros(4)
f1 = np.zeros(4)
ndcg = np.zeros(4)
RSP = np.zeros(4)
REO = np.zeros(4)

precision 

array([0., 0., 0., 0.])

In [26]:
len(user_type)

943

In [27]:
user_type.shape

(943, 2)

In [28]:
#tf.compat.v1.disable_eager_execution()
#tf.enable_eager_execution()
for i in range(n):
    with tf.compat.v1.Session() as sess:
        fairadvbpr = FairAdvBPR(sess, dict_args, train_df, test_df, user_type, type_error_weight, key_type, user_type_list, item_type_count)
        [prec_one, rec_one, f_one, ndcg_one, Rec] = fairadvbpr.run()
        #[RSP_one, REO_one] = utility.ranking_analysis(Rec, vali_df, train_df, key_genre, item_genre_list, user_genre_count)
#         precision += prec_one
#         recall += rec_one
#         f1 += f_one
#         ndcg += ndcg_one
#         RSP += RSP_one
#         REO += REO_one

**********fairAdvBPR**********
shape q pos (?, 20)
shape p (?, 20)
shape predict pos  (?,)
shape adv_last  (?, 1)
INFO:tensorflow:Restoring parameters from ./ml1m-6/BPR_check_points\check_point.ckpt-41
Training // Epoch 1 //  Total r_cost = 196541.26508  Total a_cost = 0.00000 total pre adversarial accuracy = 0.00000 total post adversarial accuracy = 0.00000 Training time : 406 ms negative Sampling time : 7894 ms negative samples : 400730
precision_1	[0.4316013],	||	 precision_5	[0.3329799],	||	 precision_10	[0.2765642],	||	 precision_15	[0.2465182]
recall_1   	[0.0286549],	||	 recall_5   	[0.1123112],	||	 recall_10   	[0.1761304],	||	 recall_15   	[0.2304581]
f_measure_1	[0.0537417],	||	 f_measure_5	[0.1679681],	||	 f_measure_10	[0.2152063],	||	 f_measure_15	[0.2382178]
ndcg_1     	[0.4316013],	||	 ndcg_5     	[0.3560628],	||	 ndcg_10     	[0.3325644],	||	 ndcg_15     	[0.3297950]
Metrics for user type	 M
precision_1	[0.4507463],	||	 precision_5	[0.3576119],	||	 precision_10	[0.296119

AUC global is:  0.9101228801829144
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversair

post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of m

post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of m

post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.0
post adversarial -- adversaire dejoue after one re-training of main model  0.0
boucle adversarial fin
post adversarial -- adversaire re

...

post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire re

post adversarial -- adversaire dejoue after one re-training of main model  0.49951171875
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.49951171875
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.49951171875
post adversarial -- adversaire dejoue after one re-training of main model  0.49951171875
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.49951171875
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle

post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.49951171875
post adversarial -- adversaire dejoue after one re-training of main model  0.49951171875
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.49951171875
post adversarial -- adversaire dejoue after one re-training of main model  0.49951171875
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adve

post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.49951171875
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adversaire ready after one adv epoch training  0.5
post adversarial -- adversaire dejoue after one re-training of main model  0.5
boucle adversarial fin
post adversarial -- adv

AUC global is:  0.887507892823958
precision_1	[0.4729586],	||	 precision_5	[0.3459173],	||	 precision_10	[0.2873807],	||	 precision_15	[0.2528102]
recall_1   	[0.0336382],	||	 recall_5   	[0.1163210],	||	 recall_10   	[0.1811276],	||	 recall_15   	[0.2296136]
f_measure_1	[0.0628092],	||	 f_measure_5	[0.1740983],	||	 f_measure_10	[0.2222056],	||	 f_measure_15	[0.2406542]
ndcg_1     	[0.4729586],	||	 ndcg_5     	[0.3763294],	||	 ndcg_10     	[0.3503049],	||	 ndcg_15     	[0.3430003]


In [29]:
                    #measure adversary accuracy
                    
#                     m = tf.keras.metrics.Accuracy()
#                     m.update_state(self.user_type[user_idx_list,:], adv_output)
#                     m = m.result()
#                    print('accu adv',acc_adv)
#                     accuracy_adv.append(tf.reduce_sum(m.result()))
                    
#                 #sauvegarde prediction adversaire
#                 filename = './const_IembfairAdvBPR_new_results/epoch'+ str(itr) +'_adv_' + self.dataname + '_advAccuracy.npy'
#                 os.makedirs(os.path.dirname(filename), exist_ok=True)           
#                 with open(filename, "wb") as f:
#                     np.save(f, sum(accuracy_adv)/len(accuracy_adv))

In [30]:
import matplotlib.pyplot as plt
import pandas as pd
from colour import Color

def savepdf_barplot_color_gradient(ymin = 0.5, ymax = 0.7, whis = 5, start_color='pink',end_color='blue',num_color=5, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    fig = plt.figure()
    gs = fig.add_gridspec(1, 2, hspace=0, wspace=0)
    (ax1, ax2) = gs.subplots(sharex='col', sharey='row')
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    ax1.bar(X_axis, axis_y1, color=colors)
    ax1.hlines(y=axis_y1[0], xmin = 0, xmax = len(axis_x)-1, colors='black', linestyles='--', lw=1)
    
    plt.sca(ax1)
    plt.xticks(X_axis, axis_x, rotation =50)
    #plt.xlabel(xlabel)
    #fig.suptitle(title)
    plt.ylabel(ylabel, fontsize=18)
    plt.rcParams.update({'font.size': 13}) 
    plt.grid()
    
    plt.sca(ax2)
    ax2.boxplot(axis_y1, whis = whis)
    ax1.set_ylim(ymin, ymax)
    
    plt.tight_layout()                                    
    plt.savefig(plot_file)

def savepdf_barplot_color_gradient2(start_color='pink',end_color='blue',num_color=20, title='',axis_x = None, xlabel = '', axis_y1 = None, ylabel ='', plot_file = ''):
    
    red = Color(start_color)
    colors = list(red.range_to(Color(end_color), num_color))
    colors = [color.rgb for color in colors]
    
    X_axis = np.arange(len(axis_x))
    plt.bar(X_axis, axis_y1, color=colors)
    
    
    plt.xticks(X_axis, axis_x, rotation =70)
    plt.xlabel(xlabel)
    plt.ylabel(ylabel)
    plt.title(title)
    plt.grid()
    
   # plt.tight_layout()
    plt.savefig(plot_file)